# RESISC45 Knowledge Distillation

Offline KD: SigLIP2 teacher (~93M) -> MobileNetV3-Large student (~4M) on NWPU-RESISC45.

Pipeline: cache teacher logits once (PyTorch) -> train Keras student with KD loss
`L = a*T^2*KL(teacher||student) + (1-a)*CE`. Single deterministic 224px view.

Run cells top-to-bottom. Each section has a verify print.

## 0. Install teacher deps (run once)

In [ ]:
%pip install -q torch transformers datasets tqdm

## 1. Config

In [ ]:
import os, random, numpy as np

SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 45
BATCH = 64
TEMP = 3.0          # KD temperature
ALPHA = 0.7         # weight on the soft (KD) term
EPOCHS = 15
TEACHER_ID = "prithivMLmods/RESISC45-SigLIP2"
DATASET_ID = "jonathan-roberts1/NWPU-RESISC45"
LOGITS_PATH = "teacher_logits.npy"

random.seed(SEED)
np.random.seed(SEED)

## 2. Data: load + seeded 60/20/20 split

The dataset's `label` feature gives each image an integer in the *dataset's* class
order. We do NOT use that order as canonical -- see section 3.

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, split="train")
print(ds)
dataset_names = ds.features["label"].names   # index = dataset label int -> class name
assert len(dataset_names) == NUM_CLASSES, dataset_names
ds_labels = np.array(ds["label"])

N = len(ds)
perm = np.random.default_rng(SEED).permutation(N)
n_test = int(0.2 * N); n_val = int(0.2 * N)
test_idx  = perm[:n_test]
val_idx   = perm[n_test:n_test + n_val]
train_idx = perm[n_test + n_val:]
print(f"N={N}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

## 3. Teacher + label alignment (the silent killer)

The teacher's 45 logit columns follow `teacher.config.id2label`. We make THAT the
canonical order, and remap the dataset's integer labels into it by matching class
names. If any name fails to match, the assert fires loudly here instead of
silently wrecking accuracy.

In [ ]:
import torch
from transformers import AutoImageProcessor, SiglipForImageClassification

processor = AutoImageProcessor.from_pretrained(TEACHER_ID)
teacher = SiglipForImageClassification.from_pretrained(TEACHER_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
teacher.to(device).eval()

id2label = teacher.config.id2label
teacher_names = [id2label[i] for i in range(len(id2label))]   # canonical order

def norm(s):
    return s.lower().replace("_", " ").replace("-", " ").strip()

teacher_norm = {norm(n): i for i, n in enumerate(teacher_names)}
ds2teacher = np.array([teacher_norm[norm(n)] for n in dataset_names], dtype=np.int64)
assert len(set(ds2teacher.tolist())) == NUM_CLASSES, "label alignment failed!"

labels = ds2teacher[ds_labels]   # canonical (teacher-order) label per image
print("label alignment OK")

## 4. Cache teacher logits (deterministic, single view)

Runs the teacher once over every image with its own HF processor (resize 224,
normalize 0.5). Cached to disk so a kernel restart won't recompute.

In [ ]:
if os.path.exists(LOGITS_PATH):
    teacher_logits = np.load(LOGITS_PATH)
    print("loaded cached logits", teacher_logits.shape)
else:
    from tqdm.auto import tqdm
    teacher_logits = np.zeros((N, NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for start in tqdm(range(0, N, BATCH)):
            imgs = [im.convert("RGB") for im in ds[start:start + BATCH]["image"]]
            inputs = processor(images=imgs, return_tensors="pt").to(device)
            out = teacher(**inputs).logits
            teacher_logits[start:start + len(imgs)] = out.float().cpu().numpy()
    np.save(LOGITS_PATH, teacher_logits)
    print("cached", teacher_logits.shape)

**Verify alignment:** teacher top-1 on the test split should be ~0.95.

In [ ]:
t_pred = teacher_logits.argmax(1)
print(f"teacher test top-1: {(t_pred[test_idx] == labels[test_idx]).mean():.4f}")

## 5. tf.data pipelines: (image, teacher_logits) -> label

MobileNetV3 (`include_preprocessing=True`) does its own rescaling, so we feed raw
0-255 pixels resized to 224. Generator indexes the HF dataset by id, keeping
images, logits and labels aligned.

In [ ]:
import tensorflow as tf
import keras
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(indices, training):
    idx = indices.astype(np.int64)
    def gen():
        for i in idx:
            img = np.asarray(ds[int(i)]["image"].convert("RGB"), dtype=np.uint8)
            yield img, teacher_logits[i], labels[i]
    sig = (tf.TensorSpec((None, None, 3), tf.uint8),
           tf.TensorSpec((NUM_CLASSES,), tf.float32),
           tf.TensorSpec((), tf.int64))
    d = tf.data.Dataset.from_generator(gen, output_signature=sig)
    def prep(img, tl, y):
        img = tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))
        return (img, tl), y
    d = d.map(prep, num_parallel_calls=AUTOTUNE)
    if training:
        d = d.shuffle(2048, seed=SEED)
    return d.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_idx, True)
val_ds   = make_ds(val_idx, False)
test_ds  = make_ds(test_idx, False)

## 6. Student: MobileNetV3-Large + 45-class logit head

In [ ]:
base = keras.applications.MobileNetV3Large(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights="imagenet", include_preprocessing=True)
x = keras.layers.GlobalAveragePooling2D()(base.output)
x = keras.layers.Dropout(0.2)(x)
out = keras.layers.Dense(NUM_CLASSES)(x)            # logits, no activation
student = keras.Model(base.input, out, name="student_mnv3l")
print(f"student params: {student.count_params():,}")

## 7. Distiller (KD loss in train_step)

In [ ]:
class Distiller(keras.Model):
    def __init__(self, student, T=3.0, alpha=0.7):
        super().__init__()
        self.student = student
        self.T = T
        self.alpha = alpha
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.soft_tracker = keras.metrics.Mean(name="soft_kd")   # KD term (T^2 * KL)
        self.hard_tracker = keras.metrics.Mean(name="hard_ce")   # ground-truth CE term
        self.acc = keras.metrics.SparseCategoricalAccuracy(name="acc")

    @property
    def metrics(self):
        return [self.loss_tracker, self.soft_tracker, self.hard_tracker, self.acc]

    def _loss(self, y, t_logits, s_logits):
        ce = tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(y, s_logits, from_logits=True))
        t_soft = tf.nn.softmax(t_logits / self.T)
        s_logsoft = tf.nn.log_softmax(s_logits / self.T)
        kl = tf.reduce_mean(
            tf.reduce_sum(t_soft * (tf.math.log(t_soft + 1e-8) - s_logsoft), axis=1))
        soft = (self.T ** 2) * kl
        total = self.alpha * soft + (1 - self.alpha) * ce
        return total, soft, ce

    def _update(self, y, s_logits, total, soft, ce):
        self.loss_tracker.update_state(total)
        self.soft_tracker.update_state(soft)
        self.hard_tracker.update_state(ce)
        self.acc.update_state(y, s_logits)
        return {m.name: m.result() for m in self.metrics}

    def train_step(self, data):
        (img, t_logits), y = data
        with tf.GradientTape() as tape:
            s_logits = self.student(img, training=True)
            total, soft, ce = self._loss(y, t_logits, s_logits)
        grads = tape.gradient(total, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        return self._update(y, s_logits, total, soft, ce)

    def test_step(self, data):
        (img, t_logits), y = data
        s_logits = self.student(img, training=False)
        total, soft, ce = self._loss(y, t_logits, s_logits)
        return self._update(y, s_logits, total, soft, ce)

## 8. Train (AdamW + cosine LR with warmup)

Logs to TensorBoard under `logs/kd/<timestamp>`: total loss, `soft_kd`, `hard_ce`
and `acc` for both train and validation. Launch with `tensorboard --logdir logs/kd`.

In [ ]:
import datetime

steps = (len(train_idx) + BATCH - 1) // BATCH
lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5, warmup_target=1e-3, warmup_steps=steps,
    decay_steps=steps * EPOCHS, alpha=0.01)
distiller = Distiller(student, T=TEMP, alpha=ALPHA)
distiller.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4))

logdir = os.path.join("logs", "kd", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard = keras.callbacks.TensorBoard(log_dir=logdir)
print("logging to", logdir)
hist = distiller.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=[tensorboard])

## 9. Evaluate: student top-1 + fidelity to teacher

Fidelity = how often the student agrees with the teacher (top-1) and the mean
KL between their softmax outputs on the held-out test set.

In [ ]:
s_logits_all, y_all = [], []
for (img, _tl), y in test_ds:
    s_logits_all.append(student(img, training=False).numpy())
    y_all.append(y.numpy())
s_logits_all = np.concatenate(s_logits_all)
y_all = np.concatenate(y_all)

s_pred = s_logits_all.argmax(1)
t_logits_test = teacher_logits[test_idx]
t_pred = t_logits_test.argmax(1)

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

ps, pt = softmax(s_logits_all), softmax(t_logits_test)
kl = (pt * (np.log(pt + 1e-8) - np.log(ps + 1e-8))).sum(1).mean()

print(f"student test top-1     : {(s_pred == y_all).mean():.4f}")
print(f"teacher test top-1     : {(t_pred == y_all).mean():.4f}")
print(f"top-1 agreement        : {(s_pred == t_pred).mean():.4f}")
print(f"mean KL(teacher||stud) : {kl:.4f}")

## 10. Save the trained student

In [ ]:
os.makedirs("models", exist_ok=True)
student.save("models/student_mnv3l.keras")
print("saved models/student_mnv3l.keras")

## 11. Standalone report: curves + confusion matrix

Run after training without re-running sections 1-10. Two parts:

1. **Curves** -- read the latest `logs/kd/<run>` event files and plot the scalar
   curves (`loss`, `soft_kd`, `hard_ce`, `acc`, train vs val) -> `training_curves.png`.
2. **Confusion matrix** -- load the saved student model, rebuild the test split
   internally (dataset + canonical label alignment), predict, and draw the
   normalized confusion matrix -> `confusion_matrix_standalone.png`.

In [ ]:
import glob
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

run = sorted(glob.glob(os.path.join("logs", "kd", "*")))[-1]
print("reading", run)

def read_scalars(subdir):
    path = os.path.join(run, subdir)
    ea = EventAccumulator(path, size_guidance={"scalars": 0})
    ea.Reload()
    out = {}
    for tag in ea.Tags()["scalars"]:
        name = tag[len("epoch_"):] if tag.startswith("epoch_") else tag
        ev = ea.Scalars(tag)
        out[name] = ([e.step for e in ev], [e.value for e in ev])
    return out

train = read_scalars("train")
val = read_scalars("validation")

metrics = ["loss", "soft_kd", "hard_ce", "acc"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, m in zip(axes.ravel(), metrics):
    if m in train:
        ax.plot(*train[m], label="train")
    if m in val:
        ax.plot(*val[m], label="val")
    ax.set_title(m); ax.set_xlabel("epoch"); ax.legend()
fig.tight_layout()
fig.savefig("training_curves.png", dpi=150)
plt.show()
print("saved training_curves.png")

**Confusion matrix** -- loads the model and does everything needed on its own:
dataset, canonical label alignment (via the teacher's config), the seeded test
split, prediction, and the normalized heatmap.

In [ ]:
import tensorflow as tf
import keras
from datasets import load_dataset
from transformers import AutoConfig

SEED, IMG_SIZE, NUM_CLASSES, BATCH = 42, 224, 45, 64
TEACHER_ID = "prithivMLmods/RESISC45-SigLIP2"
DATASET_ID = "jonathan-roberts1/NWPU-RESISC45"
MODEL_PATH = "models/student_mnv3s.keras"   # <- point at the saved student

# canonical class order = teacher's id2label (config only, no weights downloaded)
id2label = AutoConfig.from_pretrained(TEACHER_ID).id2label
teacher_names = [id2label[i] for i in range(len(id2label))]

def norm(s): return s.lower().replace("_", " ").replace("-", " ").strip()
teacher_norm = {norm(n): i for i, n in enumerate(teacher_names)}

ds = load_dataset(DATASET_ID, split="train")
dataset_names = ds.features["label"].names
ds2teacher = np.array([teacher_norm[norm(n)] for n in dataset_names], dtype=np.int64)
labels = ds2teacher[np.array(ds["label"])]          # canonical-order true labels

N = len(ds)
test_idx = np.random.default_rng(SEED).permutation(N)[:int(0.2 * N)]

model = keras.models.load_model(MODEL_PATH)
print("loaded", MODEL_PATH)

def load_img(i):
    img = np.asarray(ds[int(i)]["image"].convert("RGB"), dtype=np.uint8)
    return tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))

preds = []
for start in range(0, len(test_idx), BATCH):
    batch = tf.stack([load_img(i) for i in test_idx[start:start + BATCH]])
    preds.append(model(batch, training=False).numpy())
s_pred = np.concatenate(preds).argmax(1)
y_true = labels[test_idx]
print(f"student test top-1: {(s_pred == y_true).mean():.4f}")

cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.float64)
for t, p in zip(y_true, s_pred):
    cm[t, p] += 1
cm = cm / cm.sum(1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(teacher_names, rotation=90, fontsize=7)
ax.set_yticklabels(teacher_names, fontsize=7)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Student normalized confusion matrix (test)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig("confusion_matrix_standalone.png", dpi=150)
plt.show()
print("saved confusion_matrix_standalone.png")